In [2]:
import pandas as pd
import numpy as np
from Bio import SeqIO

# ============================
# UTILITY FUNCTIONS
# ============================

def load_aln_fasta(input_file):
    """Returns: A numpy array of aligned sequences and a numpy array of sequence IDs."""
    aln_sequences = SeqIO.parse(open(input_file),'fasta')
    id_array = np.array([fasta.id for fasta in aln_sequences])
    aln_sequences = SeqIO.parse(open(input_file),'fasta')
    seq_array = np.array([str(fasta.seq) for fasta in aln_sequences])
    if seq_array[0][-1] == "*":
        seq_array = np.array([seq[:-1] for seq in seq_array])
    aln_seq_array = np.zeros([len(seq_array),len(seq_array[0])], dtype=str)
    aln_id_array = []
    for i in range(len(seq_array)):
        if len(seq_array[i]) >= len(seq_array[0]):
            aln_seq_array[i,] = list(seq_array[i])[:len(seq_array[0])]
        else:
            aln_seq_array[i,] = list(seq_array[i]) + ["-"]*(len(seq_array[0])-len(seq_array[i]))
    
    assert len(aln_seq_array) == len(id_array)
    
    VALID_CHARS = set("ACDEFGHIKLMNPQRSTVWYO-")
    invalid_char_matrix = np.isin(aln_seq_array, list(VALID_CHARS), invert=True)
    rows_with_invalid_chars = np.unique(np.where(invalid_char_matrix)[0])
    num_sequences = aln_seq_array.shape[0]
    mask = np.ones(num_sequences, dtype=bool)
    mask[rows_with_invalid_chars] = False
    
    aln_seq_array_cleaned = aln_seq_array[mask, :]
    id_array_cleaned = id_array[mask]

    assert len(aln_seq_array_cleaned) == len(id_array_cleaned)

    return aln_seq_array_cleaned, id_array_cleaned

def get_HXB2_resno_array(aln_HXB2_fasta):
    """Returns: An array of HXB2 residue numbers."""
    HXB2_id = np.array([(0, 0)]*len(aln_HXB2_fasta), dtype=[("ResNo", int), ("Insertion", object)])
    res_no = 0
    ins_count = 0
    ins = False
    for i, seq in enumerate(aln_HXB2_fasta):
        if seq == "*":
            continue
        if seq != "-":
            res_no += 1
            ins_count = 0
            ins = False
            HXB2_id[i] = (int(res_no), ins_count)
        else:
            ins = True
        if ins:
            ins_count += 1
            HXB2_id[i] = (int(res_no), ins_count)
    return HXB2_id

def convert_ic_to_pic(ic_values, mw_g_mol = 145870):
    """Returns: An array of pIC50 or 80."""
    pic_array = pd.to_numeric(np.array(ic_values), errors='coerce')
    ic_g_l = pic_array * 1e-3
    ic_molar = ic_g_l / mw_g_mol
    pic_array = np.round(-np.log10(ic_molar), 2)
    return pic_array

In [3]:
# ============================
# CONFIGURATION
# ============================
bnabs = "VRC01"           # Name of the broadly neutralizing antibody
bnab_mw = 145870         # g/mol
label_name = "IC80"       # Type of neutralization label used
CUTOFF_LABEL = round(convert_ic_to_pic(1), 2)   # 1 μg/ml (IC80 < 1 µg/mL, https://doi.org/10.1073/pnas.2112887119, 10.1056/NEJMoa2031738)

# Input data paths
datafile = "../data/env_feature_020125.txt"                     # Env feature file
labelfile = "../data/VRC01_catnap.csv"                          # Neutralization label file from CATNAP
seqfile = "../data/virseqs_aa_O.fasta"                          # Aligned sequences
resfile = f"../data/selected_residues_{label_name}.csv"         # Selected residues for model trainning
inputinfofile = f"../data/input_info_{bnabs}_{label_name}.csv"   # Input Info table
inputfile = f"../data/input_{bnabs}_{label_name}.csv"           # Input table for model trainning

epitope_lst = ["V2_apex", "V3_glycan", "CD4_binding_site", "interface"]
epitopes = {
    "V2_apex": ["PGT145", "PG9", "PG16"],
    "V3_glycan": ["PGT121", "10-1074", "PGT128"],
    "CD4_binding_site": ["VRC01", "3BNC117", "NIH45-46", "N6"],
    "interface": ["8ANC195", "PGT151", "35O22"],
}
strings = " ABCDEFGHIJKLMNOPQRSTUVWXYZ"
subtype_label_dict = {
    "A": ["A1", "01_AE", "02_AG", "A1C", "A1D", "02A1", "A1CD", "A1G"],
    "B": ["B", "07_BC", "BF1", "08_BC", "14_BG", "BG", "BC"],
    "C": ["C", "CD", "41_CD", "02C"],
    "G": ["G"],
    "Others": ["D", "F1", "F2"]
}

In [4]:
bnabs_list = np.concatenate([epitopes[epi] for epi in epitope_lst])
env_feature = pd.read_csv(datafile, sep="\t")

contact_dict = dict()
for bnab in bnabs_list:
    bnab_data = env_feature[env_feature["Mab(s)(Binding type)"].str.contains(bnab, na=False)]
    # selected only the experiments with isolated bnAbs
    isolated_bnab_data = bnab_data[~(bnab_data["Mab(s)(Binding type)"].str.contains(str(bnab+"_|"+bnab+"-"), na=False))]
    
    bnab_contact_data = isolated_bnab_data[isolated_bnab_data["Feature type"].str.contains("contacts", na=False)]
    bnab_contact_resno = sorted(bnab_contact_data["Position"].unique())
    contact_dict[bnab] = bnab_contact_resno

epitope_resno_dict = dict()
for epi, abs in epitopes.items():
    epitope_resno_dict[epi] = list()
    for ab in abs:
        epitope_resno_dict[epi].extend(contact_dict[ab])
    epitope_resno_dict[epi] = sorted(set(epitope_resno_dict[epi]))

resno_all_selected = np.unique([bnab for sublist in contact_dict.values() for bnab in sublist])
print(f"Selected {len(resno_all_selected)} residues from {epitope_lst}\n{resno_all_selected}")

Selected 174 residues from ['V2_apex', 'V3_glycan', 'CD4_binding_site', 'interface']
[ 44  45  46  47  49  58  82  83  84  88  89  90  91  92  93  94  96  97
  98  99 102 105 109 117 121 122 123 124 127 134 135 136 137 139 140 156
 157 158 160 161 162 163 164 165 166 167 168 169 170 171 172 173 192 193
 197 198 199 207 230 234 236 237 238 240 241 245 246 262 275 276 277 278
 279 280 281 282 283 301 303 304 306 308 309 315 316 318 321 322 323 324
 325 326 327 328 329 330 332 334 352 353 354 355 357 362 364 365 366 367
 368 370 371 392 415 416 417 425 426 427 428 429 430 431 432 442 444 448
 455 456 457 458 459 460 461 462 463 464 465 466 467 469 470 471 472 473
 474 475 476 477 480 487 512 513 514 515 516 517 519 521 522 542 543 549
 550 554 592 611 625 637 638 640 641 643 644 647]


In [73]:
selected_res_no = resno_all_selected.copy()

aln_seq_array, id_array = load_aln_fasta(seqfile)
accession_array = np.array([code.split(' ')[0].split(".")[-1] for code in id_array])

mask_hxb2 = np.where(id_array == "B.FR.1983.HXB2.K03455") # HXB2: reference sequence
aln_HXB2_fasta = aln_seq_array[mask_hxb2][0]
HXB2_id = get_HXB2_resno_array(aln_HXB2_fasta)

mask_resno = np.where(np.isin(HXB2_id["ResNo"], selected_res_no))[0]
selected_res_no_HXB2_id = HXB2_id[mask_resno]
selected_res_seq_array = aln_seq_array[:,mask_resno]
print("Before consistant gap removal:", selected_res_seq_array.shape[1])
seq_df = pd.DataFrame({'Accession': accession_array, 'Seq name': id_array, 'Sequence': [''.join(seq) for seq in selected_res_seq_array]})
seq_df.drop_duplicates(inplace=True)
seq_df["Seq_no"]  = [_+1 for _ in range(len(seq_df))]

label_df = pd.read_csv(labelfile)
label_col = f"{bnabs}:{label_name}: geometric mean"
label_df.rename(columns={label_col: label_name}, inplace=True)
label_df = label_df[["Accession", "Seq name", "Subtype", label_name]]
label_df = label_df[(label_df[label_name].notnull()) & 
                    (label_df[label_name] != "-")
                    ]
label_df["p"+label_name] = convert_ic_to_pic(label_df[label_name], mw_g_mol=bnab_mw)
label_df.loc[label_df["p"+label_name].isna(), "p"+label_name] = 0
label_df['Label'] = np.array(label_df["p"+label_name] > CUTOFF_LABEL, dtype=int)
label_df.drop_duplicates(inplace=True)

input_df_1 = pd.merge(label_df, seq_df[["Accession","Sequence", "Seq_no"]], on=['Accession'])
input_df_2 = pd.merge(label_df, seq_df[["Seq name","Sequence", "Seq_no"]], on=['Seq name'])
info_df = pd.merge(input_df_1, input_df_2, how="outer")

selected_seq = np.array([list(seq) for seq in info_df["Sequence"]])
res_to_keep = np.where(~(np.all(selected_seq == "-", axis=0)))[0]
selected_seq = selected_seq[:,res_to_keep]
print("After consistant gap removal:", selected_seq.shape[1])
selected_res_no_HXB2_id = selected_res_no_HXB2_id[res_to_keep]

input_seqs = [''.join(seq) for seq in selected_seq]
info_df["Sequence"] = input_seqs
info_df.drop_duplicates(inplace=True)
info_df.sort_values(by="Seq_no", inplace=True)
info_df.rename(columns={"Seq name": "Seq_name"}, inplace=True)
print("Total number of sequences:", len(info_df))
print(info_df['Label'].value_counts() / len(info_df))

info_df["Subtype_Label"] = np.empty(len(info_df))
info_df["CRF"] = np.ones(len(info_df), dtype=int)
for subtype_label, subtype_lst in subtype_label_dict.items():
    info_df.loc[info_df["Subtype"].isin(subtype_lst), "Subtype_Label"] = subtype_label
    if subtype_label == "Others":
        info_df.loc[info_df["Subtype"].isin(subtype_lst), "CRF"] = 0
    else:
        info_df.loc[info_df["Subtype"] == subtype_lst[0], "CRF"] = 0
#info_df.sort_values(by=["Subtype_Label", "CRF", "Subtype", "Seq_name"], inplace=True)
print(info_df['Subtype_Label'].value_counts() / len(info_df))

info_df.to_csv(inputinfofile, index=False)

input_df = info_df[["Seq_no", "Sequence", "p"+label_name, "Label"]].copy()
input_df.rename(columns={"p"+label_name: "Value"}, inplace=True)
input_df.to_csv(inputfile, index=False)

selected_resno_df = pd.DataFrame(selected_res_no_HXB2_id)
selected_resno_df['ResLabel'] = [f"{resno}{strings[ins]}".strip() for resno, ins in selected_res_no_HXB2_id]
selected_resno_df['ResPos'] = np.arange(len(selected_resno_df))
epi_lst = []
for resno in selected_resno_df['ResNo']:
    epi_lst.append([epi for epi, res_list in epitope_resno_dict.items() if resno in res_list][0])
selected_resno_df['Epitope'] = epi_lst
print(selected_resno_df['Epitope'].value_counts())
selected_resno_df.to_csv(resfile, index=False)

Before consistant gap removal: 233
After consistant gap removal: 209
Total number of sequences: 889
0    0.655793
1    0.344207
Name: Label, dtype: float64
C         0.428571
B         0.407199
A         0.111361
Others    0.041620
G         0.011249
Name: Subtype_Label, dtype: float64
CD4_binding_site    86
interface           62
V3_glycan           32
V2_apex             29
Name: Epitope, dtype: int64
